# Data Strategy







## Technische Entscheidungen

### Pandas vs. Polars

Bei 88 Mio Zeilen und 16 GB RAM war Pandas an der Grenze.
Ein Benchmark auf den echten Ist-Daten hat die Entscheidung getroffen:

| | Pandas | Polars |
| :--- | ---: | ---: |
| Ladezeit | 25.7s | 6.6s |
| RAM Verbrauch | 6.1 GB | ~1.4 GB |
| Faktor | — | 4x schneller |

**Entscheidung: Polars** — nicht als Kompromiss, sondern weil es
der richtige Tool für diese Datenmenge ist. In der Praxis ist das
genau der Entscheidungsprozess den Data Professionals durchlaufen.

Pandas bleibt für kleine Datensätze (Events, Wetter, GTFS).
Der Wechsel zwischen beiden ist jederzeit möglich:
`df.to_pandas()` / `pl.from_pandas(df)`









## Train / Validation / Test Split

Der Split wird erst kurz vor dem Modell final entschieden —
das ist eine ML-Entscheidung, keine Wrangling-Entscheidung.
Die Daten bleiben vollständig erhalten, die Aufteilung erfolgt zur Laufzeit.

Folgende Strategien stehen zur Auswahl:

#### Option A — Chronologischer Jahres-Split (bevorzugt)
2023 → Training
2024 → Validation
2025 → Test

- Zeitliche Reihenfolge bleibt erhalten — kein Data Leakage
- Jeder Split enthält alle Jahreszeiten
- 2025 wird einmal angefasst, fertig — sauberster Test
- Nachteil: Modell sieht nur 1 Jahr zum Trainieren

#### Option B — Klassischer 80/20 Split

80% zufällig → Training
20% zufällig → Test

- Mehr Trainingsdaten
- Aber: zufälliger Split bei Zeitreihendaten ist problematisch —
  das Modell sieht "die Zukunft" beim Training (Data Leakage)
- Nur geeignet wenn Zeitabhängigkeit keine Rolle spielt

#### Option C — Rolling Window

Train: Monat 1–10 → Test: Monat 11–12
Train: Monat 2–11 → Test: Monat 12–13
...

- Simuliert echten Produktionseinsatz
- Robusteste Methode für Zeitreihendaten
- Aufwändiger zu implementieren
- Gibt ein realistisches Bild der Modell-Performance über die Zeit

#### Option D — Bi-Weekly Sampling + Event Whitelist ⭐

Gerade Kalenderwochen  → Training
Ungerade Kalenderwochen → Validation

Event-Tage immer in Training (Whitelist)

- Saisonalität perfekt erhalten — jede Woche beider Split-Gruppen
  enthält Montage, Freitage, Wetter-Muster
- Event-Tage (Street Parade, FCZ-Spiele, Züri Fäscht) werden
  gezielt in den Trainingsdaten gehalten — das Modell lernt
  genau diese kritischen Ausnahmesituationen
- Kein Data Leakage weil Wochen nie überlappen
- Flexibel: Whitelist kann jederzeit erweitert werden

```python
# Event-Whitelist Join
event_dates = pl.read_csv("data/interim/events.csv", separator=";")["Datum"]

df = df.with_columns(
    pl.col("BETRIEBSTAG").is_in(event_dates).alias("IS_EVENT")
)

df_train = df.filter(
    (pl.col("KW") % 2 == 0) | (pl.col("IS_EVENT"))
)
df_val = df.filter(
    (pl.col("KW") % 2 != 0) & (~pl.col("IS_EVENT"))
)
```

**Idee:** Option A für den ersten Modelldurchlauf —
einfach, sauber, nachvollziehbar. Option D als Verfeinerung
wenn das Modell auf Event-Tagen schwächelt.